# PredictShield AI - Fusion Layer v2 (Realistic Evaluation)

Trains the 5-model fusion meta-learner the **honest way**:

- **Temporal split** - per (capture day, attack campaign) 70/15/15: test traffic comes AFTER training traffic WITHIN every attack burst, and every class stays present in train/val/test
- **Causal GNN risk** - only graph windows strictly BEFORE each flow are used (no future information)
- **Label feature ablated** - GNN node feature #6 (`is_attack` rate, derived from the Label column) is zeroed at scaled mean; the deployment pipeline has no labels
- **Best-checkpoint + early stopping + temperature calibration** instead of last-epoch blind training
- **Baselines** (each model alone, soft-vote, LightGBM meta) evaluated on the SAME temporal test set

**Known residual leakage (documented, not hidden):** the base models (XGBoost / LSTM / Transformer / IF / GNN) were trained on the FULL dataset (all days), so fusion test flows were still seen by them. v2 removes fusion-level leakage only. Fully clean numbers require retraining every base model on the same temporal split - explicitly out of scope this round.

v1 reference (random split, leaky GNN map): Accuracy 0.9995, macro F1 0.9911 - inflated.

In [ ]:
import sys, subprocess
try:
    import torch, torch_geometric
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "--no-deps"])
    import torch, torch_geometric
try:
    import onnxruntime
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "onnx"])
    import onnxruntime
try:
    import onnxscript  # required by torch.onnx.export default (dynamo) path in PyTorch 2.11
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxscript"])
try:
    import lightgbm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm

import bisect
import gc
import json
import math
import os
import re

import joblib
import numpy as np
import pandas as pd
import onnxruntime as ort
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch_geometric.nn import GATConv
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

print(f'PyTorch {torch.__version__}  PyG {torch_geometric.__version__}  onnxruntime {onnxruntime.__version__}')

In [ ]:
WINDOW = 50
STRIDE = 10
N_FEATURES = 77
N_CLASSES = 15
FUSION_DIM = 47  # 15+15+15+1+1
NODE_FEAT_DIM = 16
GNN_HIDDEN = 32
EPOCHS = 80
PATIENCE = 10
LR = 0.001
BATCH_SIZE = 256
RANDOM_STATE = 42
LOOKBACK_SECONDS = 60  # causal GNN risk window

torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
XGB_ONNX = '/kaggle/input/models/pritammandal143/xgboost-cicids2017-v2/onnx/default/1/xgboost_cicids2017_v2.onnx'
LSTM_ONNX = '/kaggle/input/models/pritammandal143/lstm-cicids2017/onnx/default/1/lstm_cicids2017.onnx'
TRANS_ONNX = '/kaggle/input/models/pritammandal143/transformer-cicids2017/onnx/default/1/transformer_cicids2017.onnx'
IF_PKL = '/kaggle/input/models/pritammandal143/isolation-forest-cicids2017-v2/scikitlearn/default/1/isolation_forest_cicids2017_v2.pkl'
GNN_PT = '/kaggle/input/models/pritammandal143/gat-attack-path-state/pytorch/default/1/gat_attack_path_state.pt'

TR_CSV = '/kaggle/input/datasets/pritammandal143/training/cicids2017_training_ready.csv'
GEN_CSV = '/kaggle/input/datasets/pritammandal143/cicids2017-generated/cicids2017_generated_combined.csv'

for f in [XGB_ONNX, LSTM_ONNX, TRANS_ONNX, IF_PKL, GNN_PT, TR_CSV, GEN_CSV]:
    print(f'{"OK" if os.path.exists(f) else "MISSING"}  {f}')

In [ ]:
# Sanity: training_ready.csv and generated_combined.csv MUST be row-aligned
# (everything below depends on it). Spot-check labels on sampled rows.
LABEL_MAP = {
    "BENIGN": 0, "Bot": 1, "DDoS": 2, "DoS GoldenEye": 3, "DoS Hulk": 4,
    "DoS Slowhttptest": 5, "DoS slowloris": 6, "FTP-Patator": 7, "Heartbleed": 8,
    "Infiltration": 9, "PortScan": 10, "SSH-Patator": 11,
    "Web Attack Brute Force": 12, "Web Attack Sql Injection": 13, "Web Attack XSS": 14,
}

tr_labels = pd.read_csv(TR_CSV, usecols=['label'])['label'].astype(np.int32).values
gen_raw = pd.read_csv(GEN_CSV, usecols=['Label'])['Label'].astype(str)
gen_raw = gen_raw.str.strip().str.replace(r'\s+', ' ', regex=True)
gen_int = gen_raw.map(LABEL_MAP)
# generated CSV writes web attacks as 'Web Attack - Brute Force' (dash variant)
alias = gen_raw.str.replace(' - ', ' ', regex=False)
gen_int = gen_int.fillna(alias.map(LABEL_MAP))
n_unmapped = int(gen_int.isna().sum())
gen_labels = gen_int.fillna(-1).astype(np.int32).values

if len(tr_labels) != len(gen_labels):
    raise RuntimeError(
        f'Row count mismatch: training_ready {len(tr_labels):,} vs generated {len(gen_labels):,}')

rng = np.random.RandomState(RANDOM_STATE)
sample = rng.choice(len(tr_labels), size=min(5000, len(tr_labels)), replace=False)
match = float((tr_labels[sample] == gen_labels[sample]).mean())
print(f'Row-order alignment: {match:.4f} on {len(sample):,} sampled rows '
      f'({n_unmapped:,} unmapped generated labels)')
if n_unmapped:
    print('Top unmapped generated labels:')
    print(gen_raw[gen_int.isna()].value_counts().head(5).to_string())
assert match >= 0.99, 'training_ready and generated row order do NOT match - STOP and fix'
print('Alignment OK')
del tr_labels, gen_raw, gen_int, gen_labels, sample
_ = gc.collect()

In [ ]:
# Load training_ready (scaled 77 features + label), downsample BENIGN to 200k.
# keep_idx stays SORTED, so position order = time order (chronological merge).
tr = pd.read_csv(TR_CSV, dtype=np.float32)
tr['label'] = tr['label'].astype(np.int32)
feature_cols = [c for c in tr.columns if c != 'label']

BENIGN_TARGET = 200_000
y_all = tr['label'].values
benign_idx = np.where(y_all == 0)[0]
attack_idx = np.where(y_all != 0)[0]
np.random.seed(RANDOM_STATE)
benign_sampled = np.random.choice(benign_idx, size=BENIGN_TARGET, replace=False)
keep_idx = np.sort(np.concatenate([benign_sampled, attack_idx]))

X = tr[feature_cols].values[keep_idx].astype(np.float32)
y = y_all[keep_idx].astype(np.int64)
print(f'Downsampled: {len(X):,} rows, original indices kept (sorted = time order)')
del tr
_ = gc.collect()

In [ ]:
# Temporal zones per ORIGINAL row, derived from generated CSV Timestamp.
# CICIDS2017 capture days (July 2017): 3=Mon, 4=Tue, 5=Wed, 6=Thu, 7=Fri.
# Split EACH (day, class) campaign by time 70/15/15: test traffic comes
# AFTER training traffic within every attack burst, AND every class stays
# present in train/val/test. (A plain per-day cut left most classes
# train-only - CICIDS2017 attacks cluster in a few hours of each day.)
ts = pd.read_csv(GEN_CSV, usecols=['Timestamp'])['Timestamp'].astype(str)
t = pd.to_datetime(ts, format='%d/%m/%Y %H:%M:%S', errors='coerce')
t = t.fillna(pd.to_datetime(ts, format='%d/%m/%Y %H:%M', errors='coerce'))

days = t.dt.day.fillna(-1).values.astype(np.int64)
t_epoch = (t.astype('int64') // 10**9).values.astype(np.int64)
del ts, t
_ = gc.collect()
assert len(days) == len(y_all), f'timestamp rows {len(days):,} != training rows {len(y_all):,}'

ZONE_TRAIN, ZONE_VAL, ZONE_TEST = 0, 1, 2
row_zone = np.full(len(days), -1, dtype=np.int64)
MIN_ROWS_TO_SPLIT = 20  # tinier campaigns cannot be split meaningfully
day_cls = pd.DataFrame({'day': days, 'label': y_all})
for key, members in day_cls.groupby(['day', 'label']).groups.items():
    m = np.sort(np.asarray(members))  # chronological within the campaign
    n = len(m)
    if n < MIN_ROWS_TO_SPLIT:
        row_zone[m] = ZONE_TRAIN
        continue
    cut70, cut85 = int(n * 0.70), int(n * 0.85)
    row_zone[m[:cut70]] = ZONE_TRAIN
    row_zone[m[cut70:cut85]] = ZONE_VAL
    row_zone[m[cut85:]] = ZONE_TEST

print('Zone sizes (original rows):')
for z, name in [(ZONE_TRAIN, 'train'), (ZONE_VAL, 'val'), (ZONE_TEST, 'test')]:
    print(f'  {name:5s}: {(row_zone == z).sum():,}')
print(f'  other: {(row_zone == -1).sum():,} (unparsed timestamps)')

In [ ]:
# Build sequences; each 50-flow window must stay inside ONE temporal zone
# (no straddling a split boundary). Sequence -> zone of its last flow.
zone_of_kept = row_zone[keep_idx]

def build_sequences_zoned(Xa, ya, zones, window=WINDOW, stride=STRIDE):
    seqs, labels, last_orig, seq_zone = [], [], [], []
    for start in range(0, len(Xa) - window + 1, stride):
        z = zones[start + window - 1]
        if z < 0 or not np.all(zones[start:start + window] == z):
            continue
        seqs.append(Xa[start:start + window])
        labels.append(ya[start + window - 1])
        last_orig.append(keep_idx[start + window - 1])
        seq_zone.append(z)
    return np.stack(seqs), np.array(labels), np.array(last_orig), np.array(seq_zone)

X_seq, y_seq, last_orig, seq_zone = build_sequences_zoned(X, y, zone_of_kept)
print(f'Sequences: {X_seq.shape}')

# Drop classes with < 5 sequences (same rule as v1)
counts = pd.Series(y_seq).value_counts()
keep_classes = counts[counts >= 5].index.tolist()
mask = np.isin(y_seq, keep_classes)
X_seq, y_seq, last_orig, seq_zone = X_seq[mask], y_seq[mask], last_orig[mask], seq_zone[mask]
print(f'After rare-class drop: {X_seq.shape[0]:,} sequences')

def take(zone):
    m = seq_zone == zone
    return X_seq[m], y_seq[m], last_orig[m]

X_train, y_train, idx_train = take(ZONE_TRAIN)
X_val, y_val, idx_val = take(ZONE_VAL)
X_test, y_test, idx_test = take(ZONE_TEST)
print(f'Train {len(X_train):,}  Val {len(X_val):,}  Test {len(X_test):,}')

print('\nClass distribution per zone:')
dist = pd.DataFrame({
    'train': pd.Series(y_train).value_counts(),
    'val': pd.Series(y_val).value_counts(),
    'test': pd.Series(y_test).value_counts(),
}).fillna(0).astype(int).sort_index()
print(dist.to_string())

del X, X_seq, zone_of_kept
_ = gc.collect()

In [ ]:
# 4 flow models -> 46 fusion dims (XGB 15 + LSTM 15 + Transformer 15 + IF 1)
xgb_sess = ort.InferenceSession(XGB_ONNX, providers=['CPUExecutionProvider'])
lstm_sess = ort.InferenceSession(LSTM_ONNX, providers=['CPUExecutionProvider'])
trans_sess = ort.InferenceSession(TRANS_ONNX, providers=['CPUExecutionProvider'])
iso = joblib.load(IF_PKL)
print('4 flow models loaded')

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def flow_model_features(X_split, bs=1024):
    """Run the 4 flow models in batches (avoids OOM); return (N, 46)."""
    outs = []
    for i in range(0, len(X_split), bs):
        xb = X_split[i:i + bs]
        last_flow = xb[:, -1, :].astype(np.float32)
        full_seq = xb.astype(np.float32)
        xgb_out = xgb_sess.run(None, {'features': last_flow})
        xgb_proba = xgb_out[1] if xgb_out[1].shape[1] == N_CLASSES else xgb_out[0]
        lstm_proba = softmax(lstm_sess.run(None, {'input': full_seq})[0])
        trans_proba = softmax(trans_sess.run(None, {'input': full_seq})[0])
        if_scores = iso.decision_function(last_flow.astype(np.float64))
        outs.append(np.concatenate(
            [xgb_proba, lstm_proba, trans_proba, -if_scores.reshape(-1, 1)],
            axis=1).astype(np.float32))
    return np.concatenate(outs, axis=0)

print('Building flow-model features...')
F_train = flow_model_features(X_train); del X_train; _ = gc.collect()
F_val = flow_model_features(X_val)    # X_val KEPT - the IF baseline re-tunes its threshold on val
F_test = flow_model_features(X_test)  # X_test kept for the baselines cell
print(f'Flow features: train {F_train.shape}, val {F_val.shape}, test {F_test.shape}')

In [ ]:
# GNN: run the pretrained GAT on every 5s window graph, store per-IP risk TIMELINE.
#
# Two honesty fixes vs v1:
#   1. node feature #6 (is_attack rate - derived from the Label column) is kept
#      at 0.0 in scaled space (= training mean, "unknown attackness"); the
#      deployment pipeline has no labels, so the risk feature must not need it
#   2. risks are stored per (ip, window_start) so lookups can use only windows
#      strictly BEFORE a flow - no future information
# Also: node features are StandardScaler-normalized (as in Gnn.ipynb training),
# which v1 forgot entirely (train/inference skew).

class AttackPathGAT(torch.nn.Module):
    def __init__(self, in_channels=NODE_FEAT_DIM, hidden=GNN_HIDDEN, out_channels=1):
        super().__init__()
        self.g1 = GATConv(in_channels, hidden, heads=8, dropout=0.3)
        self.g2 = GATConv(hidden * 8, hidden, heads=4, dropout=0.3)
        self.g3 = GATConv(hidden * 4, out_channels, heads=1, concat=False)
    def forward(self, x, edge_index, edge_attr=None):
        x = F.elu(self.g1(x, edge_index))
        x = F.elu(self.g2(x, edge_index))
        x = self.g3(x, edge_index)
        return x.squeeze(-1)

gnn = AttackPathGAT().to(DEVICE)
gnn.load_state_dict(torch.load(GNN_PT, map_location=DEVICE))
gnn.eval()
print('GNN loaded')

WINDOW_SECONDS = 5

def build_ip_risk_timeline(csv_path):
    cols = [
        'Source IP', 'Destination IP', 'Source Port', 'Destination Port',
        'Protocol', 'Timestamp', 'Flow Duration',
        'Total Fwd Packets', 'Total Backward Packets',
        'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
        'Average Packet Size', 'Max Packet Length',
        'SYN Flag Count', 'RST Flag Count', 'FIN Flag Count', 'ACK Flag Count',
    ]
    df = pd.read_csv(csv_path, usecols=cols).dropna()
    df['t'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df['t'] = df['t'].fillna(pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M', errors='coerce'))
    df = df.dropna(subset=['t'])
    df['t'] = df['t'].astype('datetime64[s]').astype('int64')
    df['window'] = df['t'] // WINDOW_SECONDS
    df['day'] = pd.to_datetime(df['t'], unit='s').dt.day
    df = df[df['day'] != 3]  # Monday: all benign, skipped in GNN training too

    windows = []  # (win_start_epoch, ip_to_id, feat, edge_agg)
    for win_start, group in df.groupby('window'):
        if len(group) < 2:
            continue
        ips = pd.unique(group[['Source IP', 'Destination IP']].values.ravel())
        ip_to_id = {ip: i for i, ip in enumerate(ips)}
        n = len(ips)
        feat = np.zeros((n, NODE_FEAT_DIM), dtype=np.float32)
        for ip, i in ip_to_id.items():
            m = (group['Source IP'] == ip) | (group['Destination IP'] == ip)
            sub = group[m]
            ports = pd.unique(sub[['Source Port', 'Destination Port']].values.ravel())
            peers = pd.unique(sub[['Source IP', 'Destination IP']].values.ravel())
            feat[i, 0] = len(sub)
            feat[i, 1] = sub['Total Length of Fwd Packets'].sum() + sub['Total Length of Bwd Packets'].sum()
            feat[i, 2] = sub['Total Fwd Packets'].sum() + sub['Total Backward Packets'].sum()
            feat[i, 3] = sub['Average Packet Size'].mean()
            feat[i, 4] = len(ports)
            feat[i, 5] = len(peers)
            # feat[i, 6] stays 0.0  <- is_attack rate ABLATED (scaled mean)
            feat[i, 7] = sub['Protocol'].nunique()
            feat[i, 8] = sub['SYN Flag Count'].sum()
            feat[i, 9] = sub['RST Flag Count'].sum()
            feat[i, 10] = sub['FIN Flag Count'].sum()
            feat[i, 11] = sub['ACK Flag Count'].sum()
            feat[i, 12] = sub['Flow Duration'].sum()
            feat[i, 13] = sub['Flow Duration'].mean()
            feat[i, 14] = sub['Max Packet Length'].max()
            feat[i, 15] = 1 if any(p in (80, 443) for p in ports) else (0.5 if 22 in ports else 0.0)
        edge_agg = group.groupby(['Source IP', 'Destination IP']).agg(
            pkt_count=('Total Fwd Packets', 'sum'),
            bytes_sum=('Total Length of Fwd Packets', 'sum'),
            proto=('Protocol', 'first'),
        ).reset_index()
        windows.append((int(win_start) * WINDOW_SECONDS, ip_to_id, feat, edge_agg))

    # same normalization as Gnn.ipynb: one StandardScaler over ALL nodes
    scaler = StandardScaler().fit(np.concatenate([w[2] for w in windows], axis=0))

    timeline = {}
    for win_start, ip_to_id, feat, edge_agg in windows:
        x = torch.tensor(scaler.transform(feat), dtype=torch.float32).to(DEVICE)
        src_ids = np.asarray(edge_agg['Source IP'].map(ip_to_id).values, dtype=np.int64)
        dst_ids = np.asarray(edge_agg['Destination IP'].map(ip_to_id).values, dtype=np.int64)
        edge_index = torch.tensor(np.stack([src_ids, dst_ids]), dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            scores = torch.sigmoid(gnn(x, edge_index)).cpu().numpy()
        for ip, i in ip_to_id.items():
            timeline.setdefault(ip, []).append((win_start, float(scores[i])))

    for ip in timeline:
        timeline[ip].sort(key=lambda p: p[0])
    return timeline

print('Building causal IP -> GNN risk timeline (slow step)...')
ip_risk_timeline = build_ip_risk_timeline(GEN_CSV)
n_entries = sum(len(v) for v in ip_risk_timeline.values())
print(f'{len(ip_risk_timeline):,} IPs, {n_entries:,} (ip, window) risk entries')

In [ ]:
# Causal lookup: for each sequence's last-flow Source IP, average the GNN risk
# of windows STRICTLY BEFORE the flow timestamp (last LOOKBACK_SECONDS).
gen_src_ip = pd.read_csv(GEN_CSV, usecols=['Source IP'])['Source IP'].astype(str).values

timeline_times = {ip: [p[0] for p in v] for ip, v in ip_risk_timeline.items()}
timeline_risks = {ip: [p[1] for p in v] for ip, v in ip_risk_timeline.items()}

def causal_gnn_risk(orig_indices):
    risks = np.zeros(len(orig_indices), dtype=np.float32)
    for i, oi in enumerate(orig_indices):
        times = timeline_times.get(gen_src_ip[oi])
        if not times:
            continue
        t_now = t_epoch[oi]
        hi = bisect.bisect_left(times, t_now)                     # strictly before
        lo = bisect.bisect_left(times, t_now - LOOKBACK_SECONDS)  # lookback window
        if hi > lo:
            risks[i] = float(np.mean(timeline_risks[gen_src_ip[oi]][lo:hi]))
    return risks.reshape(-1, 1)

g_train = causal_gnn_risk(idx_train)
g_val = causal_gnn_risk(idx_val)
g_test = causal_gnn_risk(idx_test)
print(f'GNN causal risk: train {g_train.shape}, val {g_val.shape}, test {g_test.shape}')
print(f'Non-zero GNN risk in test: {(g_test > 0).sum()} / {len(g_test)} '
      f'(v1 leaked-future version had 11266/11347)')

In [ ]:
# Final fusion features = 46 flow + 1 GNN = 47, standardized (fit on TRAIN only)
Z_train = np.concatenate([F_train, g_train], axis=1).astype(np.float32)
Z_val = np.concatenate([F_val, g_val], axis=1).astype(np.float32)
Z_test = np.concatenate([F_test, g_test], axis=1).astype(np.float32)
print(f'Fusion features: {Z_train.shape[1]} dims')

fusion_scaler = StandardScaler().fit(Z_train)
Z_train_s = fusion_scaler.transform(Z_train).astype(np.float32)
Z_val_s = fusion_scaler.transform(Z_val).astype(np.float32)
Z_test_s = fusion_scaler.transform(Z_test).astype(np.float32)
del F_train, F_val, F_test, Z_train, Z_val, Z_test
_ = gc.collect()

class FusionMLP(nn.Module):
    def __init__(self, in_dim=FUSION_DIM, n_classes=N_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, n_classes)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# Softened class weights (same recipe as the LSTM/Transformer notebook)
class_counts = pd.Series(y_train).value_counts()
w = {c: 1.0 / math.sqrt(class_counts.get(c, 1)) for c in range(N_CLASSES)}
mean_w = float(np.mean(list(w.values())))
w = {c: min(max(v / mean_w, 0.5), 5.0) for c, v in w.items()}
class_weights = torch.tensor([w[c] for c in range(N_CLASSES)], dtype=torch.float32, device=DEVICE)

fusion = FusionMLP().to(DEVICE)
optimizer = torch.optim.Adam(fusion.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(weight=class_weights)

train_loader = DataLoader(
    TensorDataset(torch.tensor(Z_train_s), torch.tensor(y_train, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(
    TensorDataset(torch.tensor(Z_val_s), torch.tensor(y_val, dtype=torch.long)),
    batch_size=BATCH_SIZE)
test_loader = DataLoader(
    TensorDataset(torch.tensor(Z_test_s), torch.tensor(y_test, dtype=torch.long)),
    batch_size=BATCH_SIZE)
print(fusion)

In [ ]:
# Train with best-val checkpoint + early stopping (v1 saved the LAST epoch)
def evaluate_fusion(loader):
    fusion.eval()
    preds, labels = [], []
    with torch.no_grad():
        for zb, yb in loader:
            zb, yb = zb.to(DEVICE), yb.to(DEVICE)
            preds.append(fusion(zb).argmax(dim=1).cpu())
            labels.append(yb.cpu())
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

best_val_f1, best_state, bad_epochs = 0.0, None, 0
for epoch in range(EPOCHS):
    fusion.train()
    total_loss = 0.0
    for zb, yb in train_loader:
        zb, yb = zb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(fusion(zb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    val_preds, val_labels = evaluate_fusion(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().clone() for k, v in fusion.state_dict().items()}
        bad_epochs = 0
        marker = '  *best'
    else:
        bad_epochs += 1
    print(f'Epoch {epoch+1:3d}/{EPOCHS}  loss={total_loss/len(train_loader):.4f}  val_f1={val_f1:.4f}{marker}')
    if bad_epochs >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1} (no val improvement for {PATIENCE} epochs)')
        break

fusion.load_state_dict(best_state)
print(f'Restored best checkpoint (val macro F1 = {best_val_f1:.4f})')

In [ ]:
# Temperature calibration on VAL (v1 applied softmax to raw logits).
# Grid search instead of an unconstrained optimizer: LBFGS can run T -> 0,
# which makes softmax pathologically overconfident (confidence ~ 1.0 always).
fusion.eval()
val_logits, val_labels_all = [], []
with torch.no_grad():
    for zb, yb in val_loader:
        val_logits.append(fusion(zb.to(DEVICE)).cpu())
        val_labels_all.append(yb)
val_logits = torch.cat(val_logits)
val_labels_all = torch.cat(val_labels_all)

nll = nn.CrossEntropyLoss()
best_T, best_nll = 1.0, float('inf')
for T in np.geomspace(0.1, 10.0, 200):
    with torch.no_grad():
        loss = float(nll(val_logits / float(T), val_labels_all))
    if loss < best_nll:
        best_nll, best_T = loss, float(T)

TEMPERATURE = float(min(max(best_T, 0.5), 5.0))  # keep confidence meaningful
print(f'Temperature: {TEMPERATURE:.4f} (1.0 = no change, >1 = softer confidence)')

In [ ]:
# Baselines on the SAME temporal test set - proves (or disproves) fusion value
def report_row(name, preds, labels):
    return {
        'model': name,
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
    }

rows = []

f_preds, f_labels = evaluate_fusion(test_loader)
rows.append(report_row('FusionMLP v2 (47d)', f_preds, f_labels))

lgbm = lightgbm.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=63,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
lgbm.fit(Z_train_s, y_train)
rows.append(report_row('LightGBM meta (47d)', lgbm.predict(Z_test_s), y_test))

last_flow = X_test[:, -1, :].astype(np.float32)
xgb_out = xgb_sess.run(None, {'features': last_flow})
xgb_proba = xgb_out[1] if xgb_out[1].shape[1] == N_CLASSES else xgb_out[0]
rows.append(report_row('XGBoost alone', xgb_proba.argmax(axis=1), y_test))

lstm_logits = np.concatenate(
    [lstm_sess.run(None, {'input': X_test[i:i+1024].astype(np.float32)})[0]
     for i in range(0, len(X_test), 1024)], axis=0)
lstm_proba = softmax(lstm_logits)
rows.append(report_row('LSTM alone', lstm_logits.argmax(axis=1), y_test))

trans_logits = np.concatenate(
    [trans_sess.run(None, {'input': X_test[i:i+1024].astype(np.float32)})[0]
     for i in range(0, len(X_test), 1024)], axis=0)
trans_proba = softmax(trans_logits)
rows.append(report_row('Transformer alone', trans_logits.argmax(axis=1), y_test))

vote_proba = (xgb_proba + lstm_proba + trans_proba) / 3.0
rows.append(report_row('Soft-vote (XGB+LSTM+Trans)', vote_proba.argmax(axis=1), y_test))

# Isolation Forest alone (binary attack vs benign, threshold tuned on VAL)
y_val_bin = (y_val != 0).astype(int)
iso_val_scores = iso.decision_function(X_val[:, -1, :].astype(np.float64))
best_thr, best_bin_f1 = 0.0, 0.0
for thr in np.arange(-0.20, 0.25, 0.01):
    f1 = f1_score(y_val_bin, (iso_val_scores < thr).astype(int), zero_division=0)
    if f1 > best_bin_f1:
        best_bin_f1, best_thr = f1, thr
iso_test_pred = (iso.decision_function(X_test[:, -1, :].astype(np.float64)) < best_thr).astype(int)
y_test_bin = (y_test != 0).astype(int)
print(f'IF alone (binary, thr={best_thr:.2f}): acc={accuracy_score(y_test_bin, iso_test_pred):.4f} '
      f'binary_f1={f1_score(y_test_bin, iso_test_pred):.4f}')

results = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
print('\n=== Baselines on the SAME temporal test set ===')
print(results.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

del X_test
_ = gc.collect()

In [ ]:
# Honest report: per-class with support, confusion matrix, absent classes.
# Labels restricted to classes actually PRESENT in y_test, so a stray wrong
# prediction of an absent class does not create a fake zero-support row.
present = np.unique(f_labels)
print('=== FUSION v2 - temporal split test ===')
print(f'Accuracy      : {accuracy_score(f_labels, f_preds):.4f}')
print(f'F1 (macro, present classes)    : '
      f'{f1_score(f_labels, f_preds, labels=present, average="macro", zero_division=0):.4f}')
print(f'F1 (weighted) : {f1_score(f_labels, f_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(f_labels, f_preds, labels=present, digits=3, zero_division=0))
print('Confusion matrix (rows=true, cols=pred, classes present in test):')
print(confusion_matrix(f_labels, f_preds, labels=present))

absent = sorted(set(np.unique(y_train)) - set(np.unique(f_labels)))
print(f'\nClasses absent from TEST (present in train): {absent}')
n_stray = int((~np.isin(f_preds, present)).sum())
print(f'Stray predictions of absent classes: {n_stray} (counted as errors above)')
print('NOTE: base models were trained on the FULL dataset, so test flows were')
print('still seen by them. v2 removes fusion-level leakage (temporal split +')
print('causal, de-labeled GNN feature) - NOT base-model leakage.')

## v1 vs v2 - fill the v2 column after running

| Metric (test) | v1 (random split, leaky GNN map) | v2 (temporal + causal GNN) |
|---|---|---|
| Accuracy | 0.9995 | _(fill)_ |
| F1 macro | 0.9911 | _(fill)_ |
| F1 weighted | 0.9995 | _(fill)_ |

Expect v2 to be LOWER. That is the point: v1's number was optimistic (random
split + future info + label-derived GNN feature). v2 is the number you can
defend. Fully clean numbers still need base models retrained on the temporal
split (out of scope this round).

In [ ]:
# Export fusion to ONNX + numerical parity check
import onnx

fusion_cpu = fusion.cpu().eval()
dummy = torch.randn(4, FUSION_DIM)
torch.onnx.export(
    fusion_cpu, dummy, '/kaggle/working/fusion_mlp_v2.onnx',
    input_names=['features'], output_names=['logits'],
    dynamic_axes={'features': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, dynamo=False,  # legacy exporter - proven on Kaggle PyTorch 2.11
)
onnx.checker.check_model(onnx.load('/kaggle/working/fusion_mlp_v2.onnx'))

fusion_sess = ort.InferenceSession('/kaggle/working/fusion_mlp_v2.onnx',
                                   providers=['CPUExecutionProvider'])
onnx_out = fusion_sess.run(None, {'features': dummy.numpy()})[0]
with torch.no_grad():
    pt_out = fusion_cpu(dummy).numpy()
diff = float(np.max(np.abs(pt_out - onnx_out)))
print(f'ONNX parity: max abs diff = {diff:.8f}')
assert diff < 1e-4, 'ONNX export deviates from PyTorch'
print('ONNX export OK')

In [ ]:
# Deployment bundle for predictshield/models/ (see predictshield/README.md)
with open('/kaggle/working/fusion_scaler.json', 'w') as f:
    json.dump({'scaler_mean': fusion_scaler.mean_.tolist(),
               'scaler_std': fusion_scaler.scale_.tolist()}, f)
with open('/kaggle/working/fusion_calib.json', 'w') as f:
    json.dump({'temperature': TEMPERATURE}, f)
with open('/kaggle/working/label_map.json', 'w') as f:
    json.dump(LABEL_MAP, f, indent=2)

print('Deployment bundle saved to /kaggle/working:')
for name in ['fusion_mlp_v2.onnx', 'fusion_scaler.json', 'fusion_calib.json', 'label_map.json']:
    print(f'  {name}: {os.path.getsize(f"/kaggle/working/{name}"):,} bytes')